# 🌸 Classificação Botânica com Deep Learning (Oxford Flowers-102)
## Execução e Reprodução no Google Colab Pro (GPU Tesla T4)

Este notebook executa o pipeline científico de ponta a ponta para classificação botânica de grão fino (*Fine-Grained Visual Categorization*) com a rede **MobileNetV2** no benchmark oficial **Oxford Flowers-102** (.189$ imagens distribuídas em 02$ classes).

### 📌 Etapas Executadas:
1. **Diagnóstico de Hardware:** Verificação da GPU (Tesla T4 de 16 GB), CUDA e PyTorch.
2. **Clonagem e Dependências:** Preparação do ambiente e bibliotecas necessárias.
3. **Suíte de Testes de Software:** Validação dos testes unitários ().
4. **Treinamento e Avaliação Completa:** Execução do  em dois estágios (Fase 1 com backbone congelado e Fase 2 com fine-tuning dos blocos finais).
5. **Métricas Verificadas:** Extração e exibição das métricas consolidadas de teste (.149$ imagens).
6. **Renderização Visual:** Exibição das 11 figuras técnicas geradas em alta resolução (300 DPI).

### 1. Diagnóstico de Hardware e Ambiente
Verifica a GPU alocada pelo Google Colab Pro, versão do CUDA e VRAM disponível.

In [ ]:
!nvidia-smi
import torch, platform
print(f"Python: {platform.python_version()}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

### 2. Clonagem do Repositório e Instalação de Dependências

In [ ]:
%cd /content
import os
if os.path.exists("/content/classificacao-botanica"):
    !rm -rf /content/classificacao-botanica

!git clone https://github.com/kelryana/classificacao-botanica.git
%cd /content/classificacao-botanica

!pip install -q -r requirements.txt

### 3. Execução da Suíte de Testes de Software
Executa a bateria de testes unitários que valida a integridade matemática das métricas, calibração ECE, McNemar e invariância do BatchNorm.

In [ ]:
!pytest test_pipeline.py -v

### 4. Execução do Pipeline Científico de Ponta a Ponta
O script executa:
- Download dos 3 splits oficiais do Oxford Flowers-102 (1.020 Treino / 1.020 Validação / 6.149 Teste);
- Extração determinística de características e benchmarking clássico (KNN, Random Forest, Regressão Logística);
- Fase 1 (Cabeça MLP com backbone e BatchNorm em eval);
- Fase 2 (Fine-tuning progressivo dos blocos features.16-18);
- Avaliação com TTA (Test-Time Augmentation) no conjunto de teste completo;
- Cálculo de McNemar, Bootstrap pareado, calibração (ECE, Brier, NLL) e Grad-CAM;
- Renderização das 11 figuras técnicas em alta resolução (300 DPI).

In [ ]:
!python pipeline_botanico.py --base-dir ./resultados_botanico --data-dir ./data --epochs-p1 8 --epochs-p2 4 --batch-size 32

### 5. Verificação e Rastreabilidade das Métricas Gravadas

In [ ]:
import json
with open("./resultados_botanico/metrics.json", "r", encoding="utf-8") as f:
    res = json.load(f)

print("=" * 65)
print("MÉTRICAS VERIFICADAS DO MODELO FINAL (TESTE - 6.149 AMOSTRAS)")
print("=" * 65)
print(f"Top-1 Accuracy: {res['metrics_ft']['top1']:.2f}%")
print(f"Top-5 Accuracy: {res['metrics_ft']['top5']:.2f}%")
print(f"Macro F1-Score: {res['metrics_ft']['f1_macro']:.2f}%")
print(f"Expected Calibration Error (ECE): {res['calibration']['ece_percentage']:.2f}%")
print(f"Brier Score: {res['calibration']['brier_score']:.4f}")
print(f"Negative Log-Likelihood (NLL): {res['calibration']['nll']:.4f}")
print(f"McNemar chi2: {res['mcnemar_ft']['chi2']:.2f} (p = {res['mcnemar_ft']['p_value']:.4e})")
print(f"Bootstrap IC 95%: [{res['boot_ci_ft']['ci_low']:.2f}%, {res['boot_ci_ft']['ci_high']:.2f}%]")
if res.get('metrics_tta'):
    print(f"Com TTA (Full Test): Top-1 = {res['metrics_tta']['top1']:.2f}%")

### 6. Inspeção e Visualização das 11 Figuras Técnicas Geradas

In [ ]:
from pathlib import Path
from IPython.display import Image, display

plots = sorted(list(Path("./resultados_botanico/plots").glob("*.png")))
print(f"Total de figuras geradas: {len(plots)}
")

for p in plots:
    print(f"📊 Exibindo: {p.name}")
    display(Image(filename=str(p), width=800))
    print("-" * 60)